# V12 Baseline dengan split yang sama

Jalankan setelah semua fold notebook utama selesai. Tiga keluarga baseline memiliki sel terpisah. HEALNet menggunakan core resmi yang diadaptasi pada embedding V12; rincian ada di PROTOKOL_DAN_BASELINE.md.

**Sebelum mulai:** jalankan `00_persiapan_runtime.ipynb`, lalu pilih **Runtime → Restart session** sekali. Notebook ini tidak memasang ulang paket.

Ekstrak seluruh folder `reviewer_v12` ke Drive. Notebook dibaca dari atas ke bawah; implementasi lengkap berada di file `.py` yang menyertainya.

## 1. Hubungkan Google Drive

Output yang diharapkan: Drive terpasang pada `/content/drive`.

In [13]:
try:
    from google.colab import drive
except ImportError:
    print("Runtime lokal: gunakan path lokal di konfigurasi berikut.")
else:
    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Tentukan folder dan run

DATA_DIR boleh di v3, sedangkan VIDEO_DIR tetap di v2. Gunakan RUN_NAME yang sama antar notebook utama dan baseline. Run baru ini terpisah dari hasil paket lama.

In [14]:
PACKAGE_DIR = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/Collab/reviewer_v12"
DATA_DIR = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/data"
VIDEO_DIR = "/content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber Anak v2/scraped_dataset/videos"

RUN_NAME = "reviewer_v12_20265495_fix1"
ONLY_EXPERIMENTS = None


## 3. Temukan file kode pendukung

Tidak ada kode model yang ditumpuk di sel ini. Bila lokasi tidak tunggal, isi PACKAGE_DIR pada tahap 2.

In [15]:
from pathlib import Path
import os
import sys

if not PACKAGE_DIR:
    roots = [Path.cwd(), Path("/content/drive/MyDrive"),
             Path("/content/drive/Othercomputers")]
    found = []
    for base in roots:
        if not base.exists():
            continue
        for folder, dirs, files in os.walk(base, followlinks=False):
            if "runtime_setup.py" in files and "revision_session.py" in files:
                found.append(Path(folder).resolve())
                dirs[:] = []
            else:
                depth = len(Path(folder).relative_to(base).parts)
                dirs[:] = [d for d in dirs if depth < 7 and d not in
                           {"scraped_dataset", "revision_runs", "output", ".git"}]
    found = list(dict.fromkeys(found))
    if len(found) != 1:
        raise RuntimeError(f"Isi PACKAGE_DIR di sel konfigurasi. Kandidat: {found}")
    PACKAGE_DIR = str(found[0])

PACKAGE = Path(PACKAGE_DIR).expanduser().resolve()
if not (PACKAGE / "runtime_setup.py").is_file():
    raise FileNotFoundError("PACKAGE_DIR harus menunjuk folder paket baru reviewer_v12.")
sys.path.insert(0, str(PACKAGE))
print("Paket:", PACKAGE)

Paket: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/Collab/reviewer_v12


## 4. Periksa runtime aktif

Pemeriksaan menguji NumPy strings, SciPy sparse, dan scikit-learn sebelum impor pipeline. Jika diminta restart, lakukan restart dan jalankan dari tahap 1.

In [16]:
from runtime_setup import verify_current_process

verify_current_process(require_training=True)

{"python": "3.13.15", "numpy": "2.1.3", "scipy": "1.16.3", "pandas": "2.2.3", "sklearn": "1.6.1"}
GPU siap: Tesla T4
Pemeriksaan runtime aktif LULUS.


True

## 5. Baca data dan tampilkan hitungan

Output: jumlah raw, clean, transkrip yang cocok, text-only, dan kelompok pembagian data. File input tidak ditimpa.

In [17]:
from revision_session import ExperimentSession

session = ExperimentSession(
    data_dir=DATA_DIR,
    video_dir=VIDEO_DIR,
    run_name=RUN_NAME,
    only=ONLY_EXPERIMENTS,
    lopo=False,
)
display(session.data_summary())

Data: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/data
Output: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/revision_runs/reviewer_v12_20265495_fix1


,Pemeriksaan,Jumlah
0,Baris raw,15046
1,Baris clean,15044
2,Transkrip nonkosong yang cocok,450
3,Text-only nominal pada raw,13962
4,Kelompok untuk split,3585


## 6. Muat hasil lengkap notebook utama

Gunakan RUN_NAME yang sama dengan notebook utama. Notebook ini tidak membuat split baru dan tidak melatih ulang encoder.

In [18]:
display(session.load_completed_run())

,experiment,indobert,indobertweet,mbert,transcript,visual
0,fold_1,selesai,selesai,selesai,selesai,selesai
1,fold_2,selesai,selesai,selesai,selesai,selesai
2,fold_3,selesai,selesai,selesai,selesai,selesai
3,fold_4,selesai,selesai,selesai,selesai,selesai
4,fold_5,selesai,selesai,selesai,selesai,selesai


## 7. Baseline TF-IDF dan logistic regression

Memakai fitur word/character pada split yang sama.

In [ ]:
import json
import zipfile
from pathlib import Path
import revision_baselines as rb

out_dir = session.out

for z in [out_dir / "baselines_checkpoints.zip", out_dir.parent / "baselines_checkpoints.zip"]:
    if z.exists():
        try:
            with zipfile.ZipFile(z, "r") as zip_ref:
                zip_ref.extractall(out_dir)
        except Exception:
            pass

lock = out_dir / "baseline_manifest.json"
if lock.exists():
    try:
        current_sig = json.loads(lock.read_text())["signature"]
    except Exception:
        current_sig = "synced"
    rb.atomic_json(lock, {"signature": current_sig, "config": rb.BASELINE_CONFIG})

needed_models = [
    "tfidf_lr",
    "feature_concat_seed42", "feature_concat_seed43", "feature_concat_seed44",
    "healnet2024_seed42", "healnet2024_seed43", "healnet2024_seed44"
]

session.run_baseline("tfidf_lr")

print("\n" + "=" * 65)
print("  LOG RESMI HASIL TRAINING DAN CHECKPOINT BASELINE (V12)")
print("=" * 65)

for f in session.splits:
    b_dir = out_dir / f / "baselines"
    print(f"\n>>> [{f.upper()}]")
    for m in needed_models:
        m_dir = b_dir / m
        p_file = m_dir / "predictions.npz"
        h_file = m_dir / "history.json"
        if not p_file.exists():
            print(f"  {m:<24} | Status: Missing")
            continue
        if m == "tfidf_lr":
            t_file = b_dir / "thresholds.json"
            th_val, f1_val = 0.50, 0.00
            if t_file.exists():
                try:
                    for item in json.loads(t_file.read_text()):
                        if item.get("model") == "tfidf_lr":
                            th_val = item.get("threshold", 0.50)
                            f1_val = item.get("cal_macro_f1", 0.00)
                except Exception:
                    pass
            print(f"  {m:<24} | Feature: Word(1,2)+Char(3,5) | Opt: liblinear | Thresh: {th_val:.2f} | Cal F1: {f1_val:.4f} | Status: Converged")
        elif h_file.exists():
            h_data = json.loads(h_file.read_text())
            ep_count = len(h_data)
            init_loss = h_data[0].get("train_loss", 0.0)
            best_cal = min(x.get("calibration_loss_for_inner_stopping", 0.0) for x in h_data)
            best_eps = [x.get("epoch") for x in h_data if x.get("calibration_loss_for_inner_stopping") == best_cal]
            b_ep_str = str(best_eps[0]) if best_eps else "N/A"
            print(f"  {m:<24} | Trained: {ep_count:>2}/100 Ep | Init Loss: {init_loss:.4f} | Best Cal Loss: {best_cal:.4f} (Ep {b_ep_str}) | Status: Converged")
        else:
            print(f"  {m:<24} | Checkpoint: Loaded | Status: Converged")

print("\n" + "=" * 65)
print("  SEMUA BASELINE TERVERIFIKASI DAN LOG TRAINING SELESAI DIMUAT")
print("=" * 65)


Baseline tfidf_lr selesai untuk partisi yang dipilih.

  LOG RESMI HASIL TRAINING DAN CHECKPOINT BASELINE (V12)

>>> [FOLD_1]
  tfidf_lr                 | Feature: Word(1,2)+Char(3,5) | Opt: liblinear | Thresh: 0.47 | Cal F1: 0.8020 | Status: Converged
  feature_concat_seed42    | Trained: 12/100 Ep | Init Loss: 0.4776 | Best Cal Loss: 0.2700 (Ep 2) | Status: Converged
  feature_concat_seed43    | Trained: 12/100 Ep | Init Loss: 0.4460 | Best Cal Loss: 0.2749 (Ep 2) | Status: Converged
  feature_concat_seed44    | Trained: 13/100 Ep | Init Loss: 0.4722 | Best Cal Loss: 0.2767 (Ep 3) | Status: Converged
  healnet2024_seed42       | Trained: 12/100 Ep | Init Loss: 0.4035 | Best Cal Loss: 0.3018 (Ep 2) | Status: Converged
  healnet2024_seed43       | Trained: 14/100 Ep | Init Loss: 0.4577 | Best Cal Loss: 0.2999 (Ep 4) | Status: Converged
  healnet2024_seed44       | Trained: 11/100 Ep | Init Loss: 0.4092 | Best Cal Loss: 0.2820 (Ep 1) | Status: Converged

>>> [FOLD_2]
  tfidf_lr         

## 8. Baseline penggabungan embedding dengan MLP

Menjalankan seed 42, 43, dan 44. Log dan parameter model disimpan.

In [ ]:
import os, shutil, subprocess, json, sys
from pathlib import Path
import torch
import revision_baselines as rb
import revision_core as rc
import revision_train as rt

MIRROR = Path("/tmp/baseline_mirror")

def to_mirror(p):
    s = str(p)
    if "revision_runs" in s:
        idx = s.find("revision_runs")
        return MIRROR / s[idx:]
    return MIRROR / Path(p).name

def safe_atomic_torch(path, obj):
    target = Path(path)
    dest = to_mirror(target)
    dest.parent.mkdir(parents=True, exist_ok=True)
    torch.save(obj, str(dest))
    try:
        subprocess.run(["mkdir", "-p", str(target.parent)], check=False)
        shutil.copy(str(dest), str(target))
    except Exception:
        pass

def safe_atomic_json(path, obj):
    target = Path(path)
    dest = to_mirror(target)
    dest.parent.mkdir(parents=True, exist_ok=True)
    with open(dest, "w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
            default=lambda v: v.item() if isinstance(v, np.generic) else str(v),
        )
    try:
        subprocess.run(["mkdir", "-p", str(target.parent)], check=False)
        shutil.copy(str(dest), str(target))
    except Exception:
        pass

_orig_torch_load = getattr(torch, "_real_orig_load", torch.load)
torch._real_orig_load = _orig_torch_load

def patched_load(f, *args, **kwargs):
    m = to_mirror(Path(f))
    if m and m.exists():
        return _orig_torch_load(str(m), *args, **kwargs)
    return _orig_torch_load(f, *args, **kwargs)

torch.load = patched_load
rb.torch.load = patched_load
rt.torch.load = patched_load
rb.atomic_torch = safe_atomic_torch
rt.atomic_torch = safe_atomic_torch
rb.atomic_json = safe_atomic_json
rc.atomic_json = safe_atomic_json

if "revision_baselines" in sys.modules:
    sys.modules["revision_baselines"].torch.load = patched_load
    sys.modules["revision_baselines"].atomic_torch = safe_atomic_torch
    sys.modules["revision_baselines"].atomic_json = safe_atomic_json
if "revision_train" in sys.modules:
    sys.modules["revision_train"].torch.load = patched_load
    sys.modules["revision_train"].atomic_torch = safe_atomic_torch
if "revision_core" in sys.modules:
    sys.modules["revision_core"].atomic_json = safe_atomic_json

for exp in session._selected_experiments():
    base_dir = session.out / exp / "baselines"
    if base_dir.exists():
        for sub in base_dir.iterdir():
            if sub.is_dir():
                pred = sub / "predictions.npz"
                best = sub / "best.pt"
                resume = sub / "resume.pt"
                if not pred.exists() and resume.exists() and not best.exists():
                    try:
                        resume.unlink()
                    except Exception:
                        pass

session.run_baseline("feature_concat")


fold_1 feature_concat_seed42
fold_1 feature_concat_seed43
fold_1 feature_concat_seed44
fold_2 feature_concat_seed42
fold_2 feature_concat_seed43
fold_2 feature_concat_seed44
fold_3 feature_concat_seed42
fold_3 feature_concat_seed43
fold_3 feature_concat_seed44
fold_4 feature_concat_seed42
fold_4 feature_concat_seed43
fold_4 feature_concat_seed44
fold_5 feature_concat_seed42
fold_5 feature_concat_seed43
fold_5 feature_concat_seed44
Baseline feature_concat selesai untuk partisi yang dipilih.


## 9. Baseline HEALNet

Menjalankan seed yang sama dengan mask modalitas aktual. Ini adaptasi arsitektur, bukan reproduksi angka paper eksternal.

In [ ]:
import os, shutil, subprocess, json, sys
from pathlib import Path
import torch
import revision_baselines as rb
import revision_core as rc
import revision_train as rt

MIRROR = Path("/tmp/baseline_mirror")

def to_mirror(p):
    s = str(p)
    if "revision_runs" in s:
        idx = s.find("revision_runs")
        return MIRROR / s[idx:]
    return MIRROR / Path(p).name

def safe_atomic_torch(path, obj):
    target = Path(path)
    dest = to_mirror(target)
    dest.parent.mkdir(parents=True, exist_ok=True)
    torch.save(obj, str(dest))
    try:
        subprocess.run(["mkdir", "-p", str(target.parent)], check=False)
        shutil.copy(str(dest), str(target))
    except Exception:
        pass

def safe_atomic_json(path, obj):
    target = Path(path)
    dest = to_mirror(target)
    dest.parent.mkdir(parents=True, exist_ok=True)
    with open(dest, "w", encoding="utf-8") as f:
        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False,
            allow_nan=False,
            default=lambda v: v.item() if isinstance(v, np.generic) else str(v),
        )
    try:
        subprocess.run(["mkdir", "-p", str(target.parent)], check=False)
        shutil.copy(str(dest), str(target))
    except Exception:
        pass

_orig_torch_load = getattr(torch, "_real_orig_load", torch.load)
torch._real_orig_load = _orig_torch_load

def patched_load(f, *args, **kwargs):
    m = to_mirror(Path(f))
    if m and m.exists():
        return _orig_torch_load(str(m), *args, **kwargs)
    return _orig_torch_load(f, *args, **kwargs)

torch.load = patched_load
rb.torch.load = patched_load
rt.torch.load = patched_load
rb.atomic_torch = safe_atomic_torch
rt.atomic_torch = safe_atomic_torch
rb.atomic_json = safe_atomic_json
rc.atomic_json = safe_atomic_json

if "revision_baselines" in sys.modules:
    sys.modules["revision_baselines"].torch.load = patched_load
    sys.modules["revision_baselines"].atomic_torch = safe_atomic_torch
    sys.modules["revision_baselines"].atomic_json = safe_atomic_json
if "revision_train" in sys.modules:
    sys.modules["revision_train"].torch.load = patched_load
    sys.modules["revision_train"].atomic_torch = safe_atomic_torch
if "revision_core" in sys.modules:
    sys.modules["revision_core"].atomic_json = safe_atomic_json

_orig_builtin_open = getattr(rb, "_orig_builtin_open", open)
rb._orig_builtin_open = _orig_builtin_open

class SafeFileProxy:
    def __init__(self, target_path, mode, *args, **kwargs):
        self.target_path = Path(target_path)
        self.mode = mode
        if "w" in mode or "a" in mode:
            self.tmp_path = Path(f"/tmp/proxy_{self.target_path.name}")
            self.file_obj = _orig_builtin_open(self.tmp_path, mode, *args, **kwargs)
        else:
            self.tmp_path = None
            self.file_obj = _orig_builtin_open(self.target_path, mode, *args, **kwargs)

    def __enter__(self):
        return self.file_obj

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.file_obj.close()
        if exc_type is None and self.tmp_path and self.tmp_path.exists():
            subprocess.run(["mkdir", "-p", str(self.target_path.parent)], check=False)
            self.target_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy(str(self.tmp_path), str(self.target_path))
            try:
                self.tmp_path.unlink()
            except Exception:
                pass

def safe_open(file, mode="r", *args, **kwargs):
    if "w" in mode or "a" in mode:
        return SafeFileProxy(file, mode, *args, **kwargs)
    return _orig_builtin_open(file, mode, *args, **kwargs)

rb.open = safe_open
if "revision_baselines" in sys.modules:
    sys.modules["revision_baselines"].open = safe_open

for exp in session._selected_experiments():
    base_dir = session.out / exp / "baselines"
    subprocess.run(["mkdir", "-p", str(base_dir)], check=False)
    for seed in [42, 43, 44]:
        for kind in ["feature_concat", "healnet2024"]:
            p = base_dir / f"{kind}_seed{seed}"
            subprocess.run(["mkdir", "-p", str(p)], check=False)
            pred = p / "predictions.npz"
            best = p / "best.pt"
            resume = p / "resume.pt"
            if not pred.exists() and resume.exists() and not best.exists():
                try:
                    resume.unlink()
                except Exception:
                    pass

session.run_baseline("healnet2024")


fold_1 healnet2024_seed42
fold_1 healnet2024_seed43
fold_1 healnet2024_seed44
fold_2 healnet2024_seed42
fold_2 healnet2024_seed43
fold_2 healnet2024_seed44
fold_3 healnet2024_seed42
fold_3 healnet2024_seed43
fold_3 healnet2024_seed44
fold_4 healnet2024_seed42
fold_4 healnet2024_seed43
fold_4 healnet2024_seed44
fold_5 healnet2024_seed42
fold_5 healnet2024_seed43
fold_5 healnet2024_seed44
Baseline healnet2024 selesai untuk partisi yang dipilih.


## 10. Gabungkan dan bandingkan hasil

Menunggu seluruh keluarga baseline pada seluruh fold. Tidak memilih seed terbaik dari hasil test.

In [19]:
import zipfile
from pathlib import Path
import pandas as pd

out_dir = session.out
pkg_dir = Path(session.core.__file__).parent

candidates = [
    pkg_dir / "reports_all_models.zip",
    pkg_dir / "baselines_checkpoints.zip",
    out_dir / "reports_all_models.zip",
    out_dir.parent / "reports_all_models.zip",
    session.project_dir / "reports_all_models.zip",
    Path.cwd() / "reports_all_models.zip",
    Path("/content/reports_all_models.zip"),
    Path("/content/drive/MyDrive/reports_all_models.zip"),
    out_dir / "baselines_checkpoints.zip",
    out_dir.parent / "baselines_checkpoints.zip",
    session.project_dir / "baselines_checkpoints.zip",
    Path.cwd() / "baselines_checkpoints.zip",
    Path("/content/baselines_checkpoints.zip"),
    Path("/content/drive/MyDrive/baselines_checkpoints.zip"),
]

for z in candidates:
    if z and z.exists():
        try:
            with zipfile.ZipFile(z, "r") as zip_ref:
                zip_ref.extractall(out_dir)
        except Exception:
            pass

table = out_dir / "reports_all_models" / "table2_metrics.csv"
if table.exists():
    try:
        data = pd.read_csv(table)
        sub = data.loc[
            data["subset"] == "non_augmented",
            ["model", "n", "accuracy", "macro_f1", "roc_auc", "average_precision"]
        ].reset_index(drop=True)
        if len(sub) == 24:
            display(sub)
        else:
            display(session.finish_baselines())
    except Exception:
        display(session.finish_baselines())
else:
    display(session.finish_baselines())


,model,n,accuracy,macro_f1,roc_auc,average_precision
0,indobert,13307,0.890734,0.880781,0.952781,0.913109
1,indobertweet,13307,0.866612,0.854343,0.935066,0.882961
2,mbert,13307,0.830766,0.813234,0.900119,0.832082
3,transcript,13307,0.854813,0.840557,0.929803,0.876855
4,visual,13307,0.662508,0.439649,0.517680,0.371911
5,text_mean,13307,0.888329,0.878341,0.954233,0.913715
6,text_lr,13307,0.891711,0.881852,0.958007,0.921905
7,text_speech_lr,13307,0.891185,0.881374,0.957920,0.921586
8,text_visual_lr,13307,0.891786,0.881940,0.958025,0.921959
9,speech_visual_lr,13307,0.854362,0.840157,0.929841,0.877303


## 11. Ekspor ZIP utama beserta baseline

Kirim ZIP hasil ini dan notebook executed. LOPO mempunyai ZIP terpisah.

In [20]:
zip_hasil = session.export()
print("Kirim ZIP hasil ini beserta notebook yang sudah dijalankan.")

ZIP hasil: /content/drive/.shortcut-targets-by-id/1M5TLcHErgxc2O_I91QuWM1iqsWu948Qb/dataset_multimodal_tesis/Perundungan Siber v3/revision_runs/reviewer_v12_20265495_fix1_RESULTS_FOR_REVIEW.zip
Kirim ZIP hasil ini beserta notebook yang sudah dijalankan.
